# Sesion 02 - Development and Ingestion
## Conectividad Segura, Formatos de Datos y Exploracion

**Curso:** Especializacion Databricks Data Engineer Associate

**Docente:** Alonso Medina

**Runtime minimo:** 13.3 LTS

**Ruta de datos:** `/Volumes/dbassociate/default/vol_landing/`

### Prerequisitos
1. Subir los tres archivos fuente al Volume antes de ejecutar este notebook:
   - `ventas_transacciones.csv`
   - `eventos_iot.json`
   - `inventario_productos.csv`
2. Desde la UI: Catalog > dbassociate > default > vol_landing > Upload

### Estructura del laboratorio
- **Lab 2A:** Configurar acceso seguro con Secret Scope
- **Lab 2B:** Lectura de CSV con esquema inferido vs explicito
- **Lab 2C:** Lectura de JSON con estructura anidada
- **Lab 2D:** Crear y leer Parquet
- **Lab 2E:** Perfilado de datos
- **Lab 2F:** Vistas y persistencia

---
## Carga inicial: verificar archivos en el Volume

In [0]:
VOLUME_PATH    = "/Volumes/dbassociate/default/vol_landing"
CSV_VENTAS     = f"{VOLUME_PATH}/ventas_transacciones.csv"
JSON_IOT       = f"{VOLUME_PATH}/eventos_iot.json"
CSV_INVENTARIO = f"{VOLUME_PATH}/inventario_productos.csv"

for ruta in [CSV_VENTAS, JSON_IOT, CSV_INVENTARIO]:
    try:
        dbutils.fs.ls(ruta)
        print(f"OK : {ruta}")
    except Exception:
        print(f"NO ENCONTRADO : {ruta}  <-- subir el archivo al Volume")

---
## Lab 2A: Configurar Acceso Seguro con Secret Scope

Este bloque demuestra el patron correcto para acceder a credenciales desde un notebook.  
No requiere ADLS real para ejecutarse; el objetivo es ver el comportamiento de `dbutils.secrets`.

In [0]:
# Verificar secret scopes disponibles en el workspace kv-braz01-secret-scope
try:
    scopes = dbutils.secrets.listScopes()
    if scopes:
        for s in scopes:
            print(f"Scope: {s.name}")
            for k in dbutils.secrets.list(s.name):
                print(f"  Key: {k.key}")
    else:
        print("No hay secret scopes configurados en este workspace.")
except Exception as e:
    print(f"Sin permisos o sin scopes: {e}")

In [0]:
# Demostracion del patron correcto con Secret Scope
# Reemplazar SCOPE_NAME y SECRET_KEY si hay un scope configurado
SCOPE_NAME = "adb-sc"    # reemplazar
SECRET_KEY = "sesion" # reemplazar

try:
    valor = dbutils.secrets.get(scope=SCOPE_NAME, key=SECRET_KEY)
    # Databricks enmascara el valor automaticamente al imprimirlo
    print(f"Valor recuperado: {valor}")
    print("El valor real nunca aparece en outputs. Databricks devuelve [REDACTED].")
except Exception:
    print("Scope no disponible. Observacion: dbutils.secrets.get() devuelve [REDACTED] al imprimirse.")
    print("Antipatron a evitar: spark.conf.set('fs.azure...key', 'clave_real') expone la clave en logs y outputs.")

In [0]:
# for letter in valor:
#     print(letter)

valor[:]

**Puntos clave:**
- `dbutils.secrets.get()` nunca imprime el valor real, devuelve `[REDACTED]`
- Antipatron: credenciales hardcodeadas en el notebook quedan en historial de Git, logs del cluster y outputs compartidos
- En Unity Catalog: usar External Locations + Storage Credentials elimina la necesidad de gestionar secretos manualmente

---
## Lab 2B: Lectura de CSV - Esquema Inferido vs Explicito

In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType
)
print(f"Leyendo archivo {CSV_VENTAS}")

# Opcion 1: inferencia de esquema
# Antipatron en produccion: Spark lee el archivo DOS veces (1x para inferir, 1x para cargar)
df_ventas_inferred = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(CSV_VENTAS)
)

print("=== Esquema INFERIDO ===")
df_ventas_inferred.printSchema()

In [0]:
df_ventas_inferred.display()

In [0]:
# Opcion 2: esquema explicito (practica recomendada para produccion)
# Una sola pasada de lectura, comportamiento predecible
schema_ventas = StructType([
    StructField("transaction_id",  StringType(),  nullable=False),
    StructField("fecha",           StringType(),  nullable=True),
    StructField("cliente_id",      StringType(),  nullable=True),
    StructField("cliente_nombre",  StringType(),  nullable=True),
    StructField("region",          StringType(),  nullable=True),
    StructField("producto_id",     StringType(),  nullable=True),
    StructField("producto_nombre", StringType(),  nullable=True),
    StructField("categoria",       StringType(),  nullable=True),
    StructField("cantidad",        IntegerType(), nullable=True),
    StructField("precio_unitario", DoubleType(),  nullable=True),
    StructField("descuento_pct",   DoubleType(),  nullable=True),
    StructField("monto_total",     DoubleType(),  nullable=True),
    StructField("canal",           StringType(),  nullable=True),
    StructField("estado",          StringType(),  nullable=True),
])

df_ventas = (
    spark.read
    .option("header", "true")
    .schema(schema_ventas)
    .csv(CSV_VENTAS)
)

print("=== Esquema EXPLICITO ===")
df_ventas.printSchema()
display(df_ventas.limit(5))

In [0]:
# Demostrar comportamiento con columna inexistente en el schema
# Spark no falla: devuelve null para la columna que no existe en el archivo
schema_con_extra = StructType([
    StructField("transaction_id", StringType(), nullable=False),
    StructField("fecha",          StringType(), nullable=True),
    StructField("monto_total",    DoubleType(), nullable=True),
    StructField("moneda",         StringType(), nullable=True),  # NO existe en el CSV
])

df_extra = (
    spark.read
    .option("header", "true")
    .schema(schema_con_extra)
    .csv(CSV_VENTAS)
)

print("=== Columna 'moneda' no existe en el CSV: Spark devuelve null sin lanzar error ===")
display(df_extra.limit(5))

# NOTA IMPORTANTE:
# nullable=False en transaction_id NO rechaza nulos en lectura.
# Spark acepta los nulos de todas formas.
# Para enforcement real se necesitan Delta Constraints (sesion 7).

---
## Lab 2C: Lectura de JSON con Estructura Anidada

In [0]:
from pyspark.sql.types import LongType

# Paso 1: inferencia para explorar la estructura
# Los campos 'lectura' y 'tags' son objetos anidados
df_iot_raw = spark.read.json(JSON_IOT)

print("=== Esquema inferido: observar campos anidados 'lectura' y 'tags' ===")
df_iot_raw.printSchema()
display(df_iot_raw.limit(3))

In [0]:
# Paso 2: esquema explicito con StructType anidado
schema_lectura = StructType([
    StructField("valor",  DoubleType(), nullable=True),
    StructField("unidad", StringType(), nullable=True),
    StructField("estado", StringType(), nullable=True),
])

schema_tags = StructType([
    StructField("linea_produccion", StringType(), nullable=True),
    StructField("turno",            StringType(), nullable=True),
])

schema_iot = StructType([
    StructField("evento_id",        StringType(),   nullable=False),
    StructField("timestamp",        StringType(),   nullable=True),
    StructField("device_id",        StringType(),   nullable=True),
    StructField("device_name",      StringType(),   nullable=True),
    StructField("ubicacion",        StringType(),   nullable=True),
    StructField("tipo_sensor",      StringType(),   nullable=True),
    StructField("fabricante",       StringType(),   nullable=True),
    StructField("lectura",          schema_lectura, nullable=True),
    StructField("bateria_pct",      IntegerType(),  nullable=True),
    StructField("senial_rssi",      IntegerType(),  nullable=True),
    StructField("firmware_version", StringType(),   nullable=True),
    StructField("tags",             schema_tags,    nullable=True),
])

df_iot = spark.read.schema(schema_iot).json(JSON_IOT)

print("=== Esquema explicito con anidado ===")
df_iot.printSchema()

In [0]:
from pyspark.sql import functions as F

# Aplanar campos anidados para analisis tabular
# col("lectura.valor") extrae el campo anidado sin explode (no es un array)
df_iot_flat = df_iot.select(
    "evento_id",
    "timestamp",
    "device_id",
    "device_name",
    "ubicacion",
    "tipo_sensor",
    "fabricante",
    F.col("lectura.valor").alias("lectura_valor"),
    F.col("lectura.unidad").alias("lectura_unidad"),
    F.col("lectura.estado").alias("lectura_estado"),
    "bateria_pct",
    "senial_rssi",
    F.col("tags.linea_produccion").alias("linea_produccion"),
    F.col("tags.turno").alias("turno"),
)

print(f"Registros totales: {df_iot_flat.count()}")
display(df_iot_flat.limit(10))

---
## Lab 2D: Crear y Leer Parquet

El archivo `inventario_productos.csv` se convierte a Parquet en este lab.  
Este patron es representativo: el origen llega en CSV, se persiste como Parquet para uso analitico.

In [0]:
schema_inventario = StructType([
    StructField("producto_id",          StringType(),  nullable=False),
    StructField("sku",                  StringType(),  nullable=True),
    StructField("nombre",               StringType(),  nullable=True),
    StructField("categoria",            StringType(),  nullable=True),
    StructField("subcategoria",         StringType(),  nullable=True),
    StructField("proveedor_id",         StringType(),  nullable=True),
    StructField("proveedor_nombre",     StringType(),  nullable=True),
    StructField("pais_origen",          StringType(),  nullable=True),
    StructField("costo_unitario",       DoubleType(),  nullable=True),
    StructField("precio_venta",         DoubleType(),  nullable=True),
    StructField("margen_pct",           DoubleType(),  nullable=True),
    StructField("stock_actual",         IntegerType(), nullable=True),
    StructField("stock_minimo",         IntegerType(), nullable=True),
    StructField("stock_maximo",         IntegerType(), nullable=True),
    StructField("ubicacion_almacen",    StringType(),  nullable=True),
    StructField("fecha_ultimo_ingreso", StringType(),  nullable=True),
    StructField("fecha_ultima_venta",   StringType(),  nullable=True),
    StructField("unidad_medida",        StringType(),  nullable=True),
    StructField("peso_kg",              DoubleType(),  nullable=True),
    StructField("activo",               StringType(),  nullable=True),
])

df_inventario_csv = (
    spark.read
    .option("header", "true")
    .schema(schema_inventario)
    .csv(CSV_INVENTARIO)
)

# Guardar como Parquet en el Volume
PARQUET_PATH = f"{VOLUME_PATH}/parquet/inventario_parquet"

(
    df_inventario_csv.write
    .format("parquet")
    .mode("overwrite")
    .save(PARQUET_PATH)
)

print(f"Parquet guardado en: {PARQUET_PATH}")

In [0]:
# Leer el Parquet: el esquema viene embebido en los metadatos del archivo
# No se necesita inferSchema ni StructType manual
df_inventario = spark.read.parquet(PARQUET_PATH)

print("=== Esquema leido desde metadatos Parquet (sin inferSchema ni StructType) ===")
df_inventario.printSchema()
print(f"Total productos: {df_inventario.count()}")
df_inventario.display()

In [0]:
# Column pruning: Parquet solo lee del disco las columnas solicitadas
# CSV lee todas las columnas siempre, sin importar el select()
df_margen = (
    spark.read
    .parquet(PARQUET_PATH)
    .select("producto_id", "nombre", "categoria",
            "costo_unitario", "precio_venta", "margen_pct", "stock_actual")
)

display(df_margen.orderBy(F.col("margen_pct").desc()))

---
## Lab 2E: Perfilado de Datos

In [0]:
# Estadisticas basicas sobre columnas numericas de ventas
print("=== describe() - dataset ventas ===")
display(df_ventas.describe(["cantidad", "precio_unitario", "descuento_pct", "monto_total"]))

In [0]:
# Conteo de nulos por columna
null_counts = df_iot_flat.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_iot_flat.columns
])

print("=== Nulos por columna - dataset IoT ===")
display(null_counts)

In [0]:
# Deteccion de duplicados sobre llave natural
df_ventas.createOrReplaceTempView("ventas_tmp")
df_iot_flat.createOrReplaceTempView("iot_tmp")

In [0]:
%sql

select count(1) from ventas_tmp 

In [0]:
# %sql -- descomentar si se ejecuta como celda SQL en Databricks
spark.sql("""
    SELECT transaction_id, COUNT(*) AS apariciones
    FROM ventas_tmp
    GROUP BY transaction_id
    HAVING COUNT(*) > 1
""").show()
# Resultado esperado: 0 filas (dataset limpio)

In [0]:
# Distribucion de ventas por categoria y estado
display(
    df_ventas
    .groupBy("categoria", "estado")
    .agg(
        F.count("*").alias("transacciones"),
        F.round(F.sum("monto_total"), 2).alias("monto_total"),
        F.round(F.avg("descuento_pct"), 1).alias("descuento_promedio"),
    )
    .orderBy(F.col("monto_total").desc())
)

In [0]:
# Alertas de stock bajo: stock_actual <= stock_minimo
display(
    df_inventario
    .filter(F.col("stock_actual") <= F.col("stock_minimo"))
    .select("producto_id", "nombre", "categoria",
            "stock_actual", "stock_minimo", "ubicacion_almacen")
    .orderBy("stock_actual")
)

In [0]:
# Distribucion de eventos IoT por estado de lectura y ubicacion
display(
    df_iot_flat
    .groupBy("ubicacion", "lectura_estado")
    .agg(
        F.count("*").alias("total_eventos"),
        F.round(F.avg("bateria_pct"), 1).alias("bateria_promedio"),
    )
    .orderBy("ubicacion", "lectura_estado")
)

---
## Lab 2F: Vistas y Persistencia

In [0]:
# Vista temporal: scope de la SparkSession actual
df_ventas.createOrReplaceTempView("ventas_tmp")

# Vista temporal global: accesible desde otros notebooks del mismo cluster
# Se accede con el prefijo global_temp.<nombre>
# df_iot_flat.createOrReplaceGlobalTempView("iot_global_tmp")

print("Vistas registradas:")
print("  ventas_tmp                  -> temporal, solo esta sesion")
print("  global_temp.iot_global_tmp  -> global, cualquier notebook de este cluster")

In [0]:
# Consulta sobre vista temporal
display(spark.sql("""
    SELECT
        region,
        canal,
        COUNT(*)                                                     AS total_transacciones,
        ROUND(SUM(monto_total), 2)                                   AS ingresos_totales,
        ROUND(AVG(descuento_pct), 1)                                 AS descuento_promedio,
        SUM(CASE WHEN estado = 'Cancelado' THEN 1 ELSE 0 END)        AS cancelaciones
    FROM ventas_tmp
    GROUP BY region, canal
    ORDER BY ingresos_totales DESC
"""))

In [0]:
# Consulta sobre vista global (simula acceso desde otro notebook)
display(spark.sql("""
    SELECT
        ubicacion,
        tipo_sensor,
        COUNT(*)                                                          AS total_eventos,
        ROUND(AVG(lectura_valor), 2)                                      AS valor_promedio,
        SUM(CASE WHEN lectura_estado = 'ALERTA' THEN 1 ELSE 0 END)        AS alertas,
        MIN(bateria_pct)                                                   AS bateria_minima
    FROM global_temp.iot_global_tmp
    GROUP BY ubicacion, tipo_sensor
    ORDER BY alertas DESC
"""))

In [0]:
VOLUME_PATH

In [0]:
DELTA_PATH = f"{VOLUME_PATH}/delta/ventas_delta"
DELTA_PATH

In [0]:
# Tabla Delta EXTERNA: Databricks controla el metadato, el cliente controla los datos
# DROP TABLE elimina solo el metadato; los archivos en la ruta permanecen intactos
DELTA_PATH = f"{VOLUME_PATH}/delta/ventas_delta"
TABLE_PATH = "abfss://metastoreuc@stacadbwus01.dfs.core.windows.net/data/e19fa4b7-a625-43aa-9bc0-d7af0ffea9a9/volumes/8c3a7bb6-717c-46b5-8a33-d2bea27242f6/delta/ventas_delta"

(
    df_ventas.write
    .format("delta")
    .mode("overwrite")
    .save(DELTA_PATH)
)

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS default.ventas_lab02
    USING DELTA
    LOCATION '{TABLE_PATH}'
""")

print(f"Tabla EXTERNA creada: default.ventas_lab02 -> {DELTA_PATH}")
print("DROP TABLE eliminara el metadato. Los archivos Delta en la ruta permanecen.")

In [0]:
# Tabla Delta MANAGED: Databricks controla metadato Y ubicacion de datos
# DROP TABLE elimina metadato Y datos fisicos
spark.sql("DROP TABLE IF EXISTS dbassociate.default.ventas_lab02_managed")

(
    df_ventas.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("dbassociate.default.ventas_lab02_managed")
)

print("Tabla MANAGED creada: default.ventas_lab02_managed")
print("ADVERTENCIA: DROP TABLE sobre tabla managed elimina los datos fisicos.")

In [0]:
# Verificar propiedades de ambas tablas
print("=== Tabla EXTERNA ===")
spark.sql("DESCRIBE DETAIL default.ventas_lab02").select("name", "location", "tableType").show(truncate=False)

print("=== Tabla MANAGED ===")
spark.sql("DESCRIBE DETAIL default.ventas_lab02_managed").select("name", "location", "tableType").show(truncate=False)

In [0]:
# Limpieza al finalizar el lab
spark.sql("DROP TABLE IF EXISTS default.ventas_lab02")
spark.sql("DROP TABLE IF EXISTS default.ventas_lab02_managed")
dbutils.fs.rm(f"{VOLUME_PATH}/ventas_delta", recurse=True)
dbutils.fs.rm(f"{VOLUME_PATH}/inventario_parquet", recurse=True)
print("Limpieza completada.")

---
## Puntos de discusion

1. Por que `nullable=False` en `transaction_id` no genero un error al leer el CSV aunque hubiera nulos?
2. Que diferencia hay entre aplanar un JSON con `col("lectura.valor")` vs usar `explode()`?
3. Si se hace DROP TABLE sobre `ventas_lab02`, los archivos en el Volume se eliminan?
4. Cuando conviene mantener el JSON anidado en la tabla vs aplanarlo en la ingesta?